# 02. Discounted REINFORCE on CartPole

A companion to the [post-training primer](https://www.norange.ai/post-training/2026/08/24/post-training-primer.html). Run all cells in order in a **Colab CPU runtime**; no GPU or account tokens are needed.

Unlike the bandit, this agent makes a sequence of decisions: push a cart left or right to balance a pole. CartPole gives +1 per step, including the step on which it fails. Its observation contains cart position and velocity and pole angle and angular velocity.

We train a small neural policy directly with REINFORCE, then plot learning and animate the policy before and after training. Training is stochastic; improvement is not a guarantee for every seed.

In [ ]:
%pip -q install "gymnasium[classic-control]==1.1.1" "torch>=2.2,<3" "matplotlib>=3.7,<4"

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # Headless rendering in Colab.
os.environ["SDL_AUDIODRIVER"] = "dummy"

import copy
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

torch.set_num_threads(1)
SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
GAMMA = 0.97
MAX_STEPS = 500
BATCH_EPISODES = 16
UPDATES = 80
LEARNING_RATE = 0.05

def make_env(render_mode=None):
    return gym.make("CartPole-v1", max_episode_steps=MAX_STEPS,
                    render_mode=render_mode)

policy = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 2))
initial_policy = copy.deepcopy(policy)

## The objective and the exact episode loss

We define a **finite-horizon task**: an episode ends at failure or after 500 steps. The time limit is part of this task, so there is no reward beyond it and no value bootstrap at truncation.

$$J_\gamma(\theta)=\mathbb E\left[\sum_{t=0}^{T-1}\gamma^t r_t\right],\qquad
G_t^{(\gamma)}=\sum_{k=t}^{T-1}\gamma^{k-t}r_k.$$

Compute the local return backward, using $G_T=0$ and $G_t=r_t+\gamma G_{t+1}$. The policy loss for an episode is

$$L(\theta)=-\sum_{t=0}^{T-1}\gamma^t G_t^{(\gamma)}\log\pi_\theta(a_t\mid s_t).$$

The **outer $\gamma^t$ matters**: the local return discounts from time $t$, while the objective discounts from the beginning of the episode. Returns are treated as fixed targets. We sum over steps and average over episodes, without return normalization, a learned baseline, or entropy bonuses, to match the derivation directly. A batch of episodes reduces sampling noise.

With $\gamma=1$, this becomes undiscounted REINFORCE. CartPole illustrates the estimator; a task with competing short- and long-term goals would better illustrate how discounting changes preferred behavior.

In [ ]:
def discounted_returns(rewards, gamma):
    returns = np.empty(len(rewards), dtype=np.float32)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = float(rewards[t]) + gamma * running
        returns[t] = running
    return torch.from_numpy(returns)

# A hand-computable trajectory checks local and episode-start discounting.
example = discounted_returns([1, 2, 3], 0.5)
torch.testing.assert_close(example, torch.tensor([2.75, 3.5, 3.0]))
torch.testing.assert_close(
    (0.5 ** torch.arange(3)) * example,
    torch.tensor([2.75, 1.75, 0.75]),
)
torch.testing.assert_close(
    discounted_returns([1, 2, 3], 1.0), torch.tensor([6., 5., 3.])
)

In [ ]:
def collect_episode(env, model, seed=None):
    observation, _ = env.reset(seed=seed)
    log_probs, rewards = [], []
    while True:
        state = torch.as_tensor(observation, dtype=torch.float32)
        distribution = Categorical(logits=model(state))
        action = distribution.sample()
        log_probs.append(distribution.log_prob(action))
        observation, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        if terminated or truncated:
            break
    return torch.stack(log_probs), rewards

@torch.no_grad()
def evaluate(model, seeds):
    # The same reset seeds and action RNG stream make comparisons reproducible.
    lengths = []
    env = make_env()
    try:
        with torch.random.fork_rng():
            torch.manual_seed(SEED + 10000)
            for seed in seeds:
                _, rewards = collect_episode(env, model, seed=int(seed))
                lengths.append(len(rewards))
    finally:
        env.close()
    return np.asarray(lengths)

EVAL_SEEDS = range(1000, 1020)
before_lengths = evaluate(initial_policy, EVAL_SEEDS)
print(f"Initial policy: {before_lengths.mean():.1f} steps on average")

## Train with complete episodes

We compare discounted REINFORCE (`GAMMA = 0.97`) with undiscounted REINFORCE (`gamma = 1.0`). Both runs start from identical weights, use fresh Adam optimizers with the same learning rate, and use matching environment and action-sampling seeds for each episode. Their trajectories can diverge as their policies learn.

Each update averages over the same number of complete episodes. Longer episodes require more environment interactions, so we compare rewards against both policy updates and cumulative environment steps. This runs two training sessions and takes roughly twice the work of the original example.

In [ ]:
def train_reinforce(gamma):
    model = copy.deepcopy(initial_policy)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {key: [] for key in
               ("steps", "discounted_return", "loss", "grad_norm", "env_steps")}
    total_steps = 0
    env = make_env()
    try:
        with torch.random.fork_rng():
            for update in range(UPDATES):
                episode_losses, lengths, objectives = [], [], []
                for episode in range(BATCH_EPISODES):
                    episode_seed = SEED + update * BATCH_EPISODES + episode
                    torch.manual_seed(episode_seed + 100000)
                    log_probs, rewards = collect_episode(env, model, seed=episode_seed)
                    returns = discounted_returns(rewards, gamma)
                    discounts = gamma ** torch.arange(len(rewards), dtype=torch.float32)
                    episode_losses.append(-(discounts * returns * log_probs).sum())
                    lengths.append(len(rewards))
                    objectives.append(returns[0].item())
                loss = torch.stack(episode_losses).mean()
                optimizer.zero_grad()
                loss.backward()
                if not torch.isfinite(loss) or any(
                    not torch.isfinite(p.grad).all() for p in model.parameters()
                ):
                    raise RuntimeError("Non-finite loss or gradient")
                grad_norm = torch.sqrt(sum(p.grad.square().sum()
                                           for p in model.parameters())).item()
                optimizer.step()
                total_steps += sum(lengths)
                history["steps"].append(float(np.mean(lengths)))
                history["discounted_return"].append(float(np.mean(objectives)))
                history["loss"].append(loss.item())
                history["grad_norm"].append(grad_norm)
                history["env_steps"].append(total_steps)
                if (update + 1) % 10 == 0:
                    print(f"gamma={gamma:g} | update {update + 1:3d}/{UPDATES}: "
                          f"reward={np.mean(lengths):6.1f}, loss={loss.item():.2f}")
    finally:
        env.close()
    return model, history

# First run: discounted reward.
assert 0 < GAMMA < 1, "Choose GAMMA below 1 for the discounted comparison."
policy, history = train_reinforce(GAMMA)

### Second run: no discounting

Start again from the initial policy, rather than continuing the discounted run. Only the discount factor changes.

In [ ]:
undiscounted_policy, undiscounted_history = train_reinforce(1.0)

## Compare learning dynamics

**Reward is the shared performance measure:** CartPole gives +1 per step, so undiscounted episode return equals episode length. Higher is better for both policies.

The raw surrogate losses have different scales because discounting changes their weights. A smaller loss for one run does **not** establish better performance, and loss need not decrease as the policy survives longer. Similarly, gradient norms show update-signal magnitude, not an estimate of gradient variance. These plots help inspect the effect of discounting; they do not by themselves prove variance reduction.

Each line below is a batch mean (loss and gradient norm are computed from the batch). A single paired run may favor either setting. Repeat with several `SEED` values before drawing a general conclusion; learning rate and return scale also affect the comparison.

In [ ]:
runs = [
    (f"gamma = {GAMMA:g}", policy, history, "#9a7bbd"),
    ("gamma = 1", undiscounted_policy, undiscounted_history, "#6b9f97"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for label, model, values, color in runs:
    updates = np.arange(1, len(values["steps"]) + 1)
    axes[0, 0].plot(updates, values["steps"], label=label, color=color)
    axes[0, 1].plot(values["env_steps"], values["steps"], label=label, color=color)
    axes[1, 0].plot(updates, values["loss"], label=label, color=color)
    axes[1, 1].plot(updates, values["grad_norm"], label=label, color=color)
for ax, xlabel, ylabel in [
    (axes[0, 0], "Policy update", "Mean undiscounted episode reward"),
    (axes[0, 1], "Cumulative training environment steps", "Mean undiscounted episode reward"),
    (axes[1, 0], "Policy update", "Raw surrogate loss (different scales)"),
    (axes[1, 1], "Policy update", "Batch gradient norm"),
]:
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    ax.legend()
axes[1, 0].set_yscale("symlog", linthresh=1)
axes[1, 1].set_yscale("symlog", linthresh=1)
fig.tight_layout()
plt.show()

print("Evaluation: stochastic policies, same 20 reset seeds and sampling stream")
print(f"Initial policy: mean reward {before_lengths.mean():.1f} ± {before_lengths.std():.1f}")
for label, model, values, color in runs:
    lengths = evaluate(model, EVAL_SEEDS)
    print(f"{label}: mean reward {lengths.mean():.1f} ± {lengths.std():.1f}; "
          f"500-step completions: {np.mean(lengths == MAX_STEPS):.0%}")
print("± denotes episode standard deviation, not uncertainty across training seeds.")

## Watch the initial policy and both trained policies

All three videos use the same reset seed and action-sampling seed. These are illustrative rollouts, not cherry-picked successes; use the evaluation averages above to assess learning. Frames are subsampled to keep the notebook small.

In [ ]:
@torch.no_grad()
def animate_policy(model, title, seed=2026):
    env = make_env(render_mode="rgb_array")
    frames = []
    try:
        observation, _ = env.reset(seed=seed)
        frames.append(env.render())
        with torch.random.fork_rng():
            torch.manual_seed(seed)
            for step in range(MAX_STEPS):
                distribution = Categorical(logits=model(torch.as_tensor(
                    observation, dtype=torch.float32)))
                observation, _, terminated, truncated, _ = env.step(
                    distribution.sample().item())
                if (step + 1) % 3 == 0 or terminated or truncated:
                    frames.append(env.render())
                if terminated or truncated:
                    break
    finally:
        env.close()
    fig, ax = plt.subplots(figsize=(6, 4))
    artist = ax.imshow(frames[0])
    ax.set_title(f"{title} — {step + 1} steps")
    ax.axis("off")
    def draw(i):
        artist.set_data(frames[i])
        return (artist,)
    animation = FuncAnimation(fig, draw, frames=len(frames), interval=60, blit=True)
    html = animation.to_jshtml(default_mode="once")
    plt.close(fig)
    return HTML(html)

plt.rcParams["animation.embed_limit"] = 60

In [ ]:
display(animate_policy(initial_policy, "Before training"))
display(animate_policy(policy, f"After training: gamma = {GAMMA:g}"))
display(animate_policy(undiscounted_policy, "After training: gamma = 1"))

## Experiments and references

- Change `SEED` and rerun from the setup cell to repeat the paired comparison from fresh initial weights. Compare multiple seeds rather than selecting the most favorable run.
- Change `GAMMA` to `0.95` and rerun both training cells. Smaller discounts focus learning on a shorter horizon but give less credit to genuinely useful delayed rewards.
- Compare reward against both update count and environment interactions. A fixed episode budget is not a fixed interaction budget.
- Keep the outer discount in the loss: dropping it no longer implements the exact episodic gradient derived in the post.

Sources: [Gymnasium CartPole documentation](https://gymnasium.farama.org/environments/classic_control/cart_pole/); [Sutton and Barto, Reinforcement Learning: An Introduction](https://incompleteideas.net/book/the-book-2nd.html), Chapters 3 and 13.